In [ ]:
from __future__ import annotations

import argparse
import csv
import re
import statistics
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List, Optional


# ============================================================
# Configuration
# ============================================================

@dataclass
class EvaluationConfig:
    """
    Controls which evaluation checks are enabled.

    You can change these settings depending on the project.
    """

    enable_exact_match: bool = True
    enable_keyword_check: bool = True
    enable_citation_check: bool = True
    enable_disclaimer_check: bool = True
    enable_latency_check: bool = True
    enable_manual_rating: bool = True

    # Minimum keyword percentage required.
    keyword_threshold: float = 0.5

    # Maximum acceptable latency in milliseconds.
    latency_threshold_ms: Optional[float] = None

    # Minimum manual rating required.
    manual_rating_threshold: Optional[float] = None


# ============================================================
# Data Models
# ============================================================

@dataclass
class EvaluationResult:
    prompt_id: str
    question: str
    expected_answer: str
    actual_answer: str

    exact_match: Optional[bool]
    keyword_score: Optional[float]

    citation_present: Optional[bool]
    disclaimer_present: Optional[bool]

    latency_ms: Optional[float]
    manual_rating: Optional[float]

    passed: bool
    failure_reasons: str


# ============================================================
# Text Utilities
# ============================================================

def normalize_text(text: str) -> str:
    """
    Normalize text for comparison.

    Example:
        "Hello, World!"
        " hello world "
        become approximately the same.
    """

    if not text:
        return ""

    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s]", "", text)

    return text.strip()


# ============================================================
# Exact Match
# ============================================================

def check_exact_match(expected: str, actual: str) -> bool:
    """
    Check whether expected and actual answers match
    after basic normalization.
    """

    return normalize_text(expected) == normalize_text(actual)


# ============================================================
# Keyword Evaluation
# ============================================================

def extract_keywords(keyword_text: str) -> List[str]:
    """
    Convert comma-separated keywords into a list.
    """

    if not keyword_text:
        return []

    return [
        keyword.strip().lower()
        for keyword in keyword_text.split(",")
        if keyword.strip()
    ]


def check_keywords(
    actual_answer: str,
    keywords: List[str]
) -> float:
    """
    Return the percentage of required keywords found.

    Example:
        Required keywords = 4
        Found = 3

        Score = 0.75
    """

    if not keywords:
        return 1.0

    actual = normalize_text(actual_answer)

    found = 0

    for keyword in keywords:
        if normalize_text(keyword) in actual:
            found += 1

    return found / len(keywords)


# ============================================================
# Citation Detection
# ============================================================

def check_citation(answer: str) -> bool:
    """
    Generic citation detection.

    Supports common citation styles such as:

        [1]
        [2]
        (Smith, 2024)
        DOI
        URL
        https://...
        Source:
        References:
    """

    if not answer:
        return False

    citation_patterns = [
        r"\[\d+\]",
        r"\([A-Za-z]+,\s*\d{4}\)",
        r"https?://",
        r"www\.",
        r"\bdoi\b",
        r"\bsource\s*:",
        r"\breferences?\s*:",
        r"\bcitation\s*:"
    ]

    return any(
        re.search(pattern, answer, re.IGNORECASE)
        for pattern in citation_patterns
    )


# ============================================================
# Disclaimer Detection
# ============================================================

def check_disclaimer(
    answer: str,
    disclaimer_keywords: Optional[List[str]] = None
) -> bool:
    """
    Check whether an answer contains a disclaimer.

    The keywords can be customized according to the domain.
    """

    if not answer:
        return False

    if not disclaimer_keywords:
        disclaimer_keywords = [
            "disclaimer",
            "not financial advice",
            "not medical advice",
            "consult a professional",
            "for informational purposes",
            "educational purposes only",
            "verify this information"
        ]

    answer_lower = answer.lower()

    return any(
        keyword.lower() in answer_lower
        for keyword in disclaimer_keywords
    )


# ============================================================
# Latency
# ============================================================

def check_latency(
    latency_ms: Optional[float],
    threshold_ms: Optional[float]
) -> bool:
    """
    Check whether response latency is within the configured limit.

    If no threshold is provided, the latency check passes
    as long as latency exists.
    """

    if latency_ms is None:
        return False

    if threshold_ms is None:
        return True

    return latency_ms <= threshold_ms


# ============================================================
# Manual Rating
# ============================================================

def check_manual_rating(
    rating: Optional[float],
    threshold: Optional[float]
) -> bool:
    """
    Check manual reviewer rating.

    Rating can normally be 1-5.
    """

    if rating is None:
        return True

    if threshold is None:
        return True

    return rating >= threshold


# ============================================================
# Single Case Evaluation
# ============================================================

def evaluate_case(
    row: dict,
    config: EvaluationConfig
) -> EvaluationResult:

    prompt_id = row.get("prompt_id", "").strip()
    question = row.get("question", "").strip()

    expected_answer = row.get("expected_answer", "").strip()
    actual_answer = row.get("actual_answer", "").strip()

    keywords = extract_keywords(
        row.get("keywords", "")
    )

    # --------------------------------------------------------
    # Optional values
    # --------------------------------------------------------

    latency_ms = None

    if row.get("latency_ms", "").strip():
        try:
            latency_ms = float(row["latency_ms"])
        except ValueError:
            latency_ms = None

    manual_rating = None

    if row.get("manual_rating", "").strip():
        try:
            manual_rating = float(row["manual_rating"])
        except ValueError:
            manual_rating = None

    # --------------------------------------------------------
    # Results
    # --------------------------------------------------------

    exact_match = None
    keyword_score = None
    citation_present = None
    disclaimer_present = None

    failures = []

    # --------------------------------------------------------
    # Exact Match
    # --------------------------------------------------------

    if config.enable_exact_match:

        exact_match = check_exact_match(
            expected_answer,
            actual_answer
        )

        if not exact_match:
            failures.append("Exact match failed")

    # --------------------------------------------------------
    # Keyword Check
    # --------------------------------------------------------

    if config.enable_keyword_check:

        keyword_score = check_keywords(
            actual_answer,
            keywords
        )

        if keyword_score < config.keyword_threshold:
            failures.append(
                f"Keyword score below threshold "
                f"({keyword_score:.2f})"
            )

    # --------------------------------------------------------
    # Citation
    # --------------------------------------------------------

    if config.enable_citation_check:

        citation_required = (
            row.get("citation_required", "false")
            .lower()
            == "true"
        )

        if citation_required:

            citation_present = check_citation(
                actual_answer
            )

            if not citation_present:
                failures.append("Citation missing")

    # --------------------------------------------------------
    # Disclaimer
    # --------------------------------------------------------

    if config.enable_disclaimer_check:

        disclaimer_required = (
            row.get("disclaimer_required", "false")
            .lower()
            == "true"
        )

        if disclaimer_required:

            disclaimer_present = check_disclaimer(
                actual_answer
            )

            if not disclaimer_present:
                failures.append("Disclaimer missing")

    # --------------------------------------------------------
    # Latency
    # --------------------------------------------------------

    if config.enable_latency_check:

        latency_passed = check_latency(
            latency_ms,
            config.latency_threshold_ms
        )

        if not latency_passed:
            failures.append("Latency check failed")

    # --------------------------------------------------------
    # Manual Rating
    # --------------------------------------------------------

    if config.enable_manual_rating:

        rating_passed = check_manual_rating(
            manual_rating,
            config.manual_rating_threshold
        )

        if not rating_passed:
            failures.append("Manual rating below threshold")

    # --------------------------------------------------------
    # Final Status
    # --------------------------------------------------------

    passed = len(failures) == 0

    return EvaluationResult(
        prompt_id=prompt_id,
        question=question,
        expected_answer=expected_answer,
        actual_answer=actual_answer,

        exact_match=exact_match,
        keyword_score=keyword_score,

        citation_present=citation_present,
        disclaimer_present=disclaimer_present,

        latency_ms=latency_ms,
        manual_rating=manual_rating,

        passed=passed,
        failure_reasons="; ".join(failures)
    )


# ============================================================
# CSV Evaluation
# ============================================================

def evaluate_csv(
    input_file: str,
    output_file: str,
    config: EvaluationConfig
):

    results = []

    with open(
        input_file,
        "r",
        encoding="latin-1", # Changed from "utf-8-sig"
        newline=""
    ) as file:

        reader = csv.DictReader(file)

        for row in reader:

            result = evaluate_case(
                row,
                config
            )

            results.append(result)

    # --------------------------------------------------------
    # Write per-case results
    # --------------------------------------------------------

    with open(
        output_file,
        "w",
        encoding="utf-8",
        newline=""
    ) as file:

        writer = csv.DictWriter(
            file,
            fieldnames=list(
                asdict(results[0]).keys()
            ) if results else []
        )

        if results:
            writer.writeheader()

            for result in results:
                writer.writerow(
                    asdict(result)
                )

    return results


# ============================================================
# Aggregate Report
# ============================================================

def calculate_percentage(
    values: List[bool]
) -> float:

    if not values:
        return 0.0

    return (
        sum(values) / len(values)
    ) * 100


def generate_report(
    results: List[EvaluationResult],
    output_file: str
):

    total = len(results)

    if total == 0:
        report = "# AI Evaluation Baseline Report\n\nNo test cases found."

        Path(output_file).write_text(
            report,
            encoding="utf-8"
        )

        return

    passed = sum(
        1 for result in results
        if result.passed
    )

    failed = total - passed

    exact_results = [
        result.exact_match
        for result in results
        if result.exact_match is not None
    ]

    keyword_results = [
        result.keyword_score
        for result in results
        if result.keyword_score is not None
    ]

    citation_results = [
        result.citation_present
        for result in results
        if result.citation_present is not None
    ]

    disclaimer_results = [
        result.disclaimer_present
        for result in results
        if result.disclaimer_present is not None
    ]

    latencies = [
        result.latency_ms
        for result in results
        if result.latency_ms is not None
    ]

    ratings = [
        result.manual_rating
        for result in results
        if result.manual_rating is not None
    ]

    lines = []

    lines.append("# AI Evaluation Baseline Report")
    lines.append("")
    lines.append("## Overall Results")
    lines.append("")
    lines.append(f"- Total test cases: {total}")
    lines.append(f"- Passed: {passed}")
    lines.append(f"- Failed: {failed}")
    lines.append(
        f"- Overall pass rate: {(passed / total) * 100:.2f}%"
    )

    lines.append("")
    lines.append("## Evaluation Metrics")
    lines.append("")

    if exact_results:
        lines.append(
            f"- Exact match: "
            f"{calculate_percentage(exact_results):.2f}%"
        )

    if keyword_results:
        lines.append(
            f"- Average keyword score: "
            f"{statistics.mean(keyword_results) * 100:.2f}%"
        )

    if citation_results:
        lines.append(
            f"- Citation presence: "
            f"{calculate_percentage(citation_results):.2f}%"
        )

    if disclaimer_results:
        lines.append(
            f"- Disclaimer presence: "
            f"{calculate_percentage(disclaimer_results):.2f}%"
        )

    if latencies:
        lines.append(
            f"- Average latency: "
            f"{statistics.mean(latencies):.2f} ms"
        )

        lines.append(
            f"- Maximum latency: "
            f"{max(latencies):.2f} ms"
        )

    if ratings:
        lines.append(
            f"- Average manual rating: "
            f"{statistics.mean(ratings):.2f}/5"
        )

    lines.append("")
    lines.append("## Failed Cases")
    lines.append("")

    failed_cases = [
        result
        for result in results
        if not result.passed
    ]

    if not failed_cases:

        lines.append("No failed cases.")

    else:

        for result in failed_cases:

            lines.append(
                f"- **{result.prompt_id}**: "
                f"{result.failure_reasons}"
            )

    Path(output_file).write_text(
        "\n".join(lines),
        encoding="utf-8"
    )


# ============================================================
# Command Line Interface
# ============================================================

def main():

    parser = argparse.ArgumentParser(
        description="Universal AI Evaluation Harness"
    )

    parser.add_argument(
        "--input",
        required=True,
        #Copy the File Path Here
        help="/content/responses.csv"
    )

    parser.add_argument(
        "--output",
        default="evaluation_results.csv",
        help="Output results CSV"
    )

    parser.add_argument(
        "--report",
        default="baseline_report.md",
        help="Baseline report file"
    )

    parser.add_argument(
        "--keyword-threshold",
        type=float,
        default=0.5,
        help="Minimum keyword score"
    )

    parser.add_argument(
        "--latency-threshold",
        type=float,
        default=None,
        help="Maximum allowed latency in milliseconds"
    )

    parser.add_argument(
        "--rating-threshold",
        type=float,
        default=None,
        help="Minimum manual rating"
    )
    # Parse arguments provided directly for Colab execution
    args = parser.parse_args(["--input", "/content/s1_ai_03_evaluation_test_data_60.csv"]) # Change the file Path Here

    config = EvaluationConfig(
        keyword_threshold=args.keyword_threshold,
        latency_threshold_ms=args.latency_threshold,
        manual_rating_threshold=args.rating_threshold
    )

    results = evaluate_csv(
        args.input,
        args.output,
        config
    )

    generate_report(
        results,
        args.report
    )

    print("\nEvaluation completed successfully.")
    print(f"Cases evaluated: {len(results)}")
    print(f"Results: {args.output}")
    print(f"Report: {args.report}")


if __name__ == "__main__":
    main()


Evaluation completed successfully.
Cases evaluated: 60
Results: evaluation_results.csv
Report: baseline_report.md
